# Predicting Product Discounts — AATD Project
**Algoritmos de Apoio à Tomada de Decisão** · FEP, Universidade do Porto · 2026/27

Authors: *[names / student numbers]*

> This notebook follows the structure required for the report:
> Abstract → Introduction → Methodology (CRISP-DM: business understanding, data understanding,
> data preparation, modeling, evaluation) → Results → Conclusions.
>
> **Expected files** (same folder as the notebook): `training.csv` (columns `i`, `d*`, `x*`, `y`)
> and `test.csv` (same columns without `y`).
> Cells marked **TODO** are narrative the authors must write after reading the outputs.

## Abstract
> **TODO — write last (150–250 words).** Problem (classify products as on discount), data
> (n products, d discrete + x continuous variables), approach (preprocessing choices, nested stratified
> cross-validation, 8 classifiers), headline result (best model, accuracy / recall / precision / F-measure)
> and the main take-away.

## 1. Introduction
> **TODO.** Suggested content: (i) why predicting promotional status matters (pricing and promotion analytics);
> (ii) the task: binary classification, `y = 1` means the product is on discount; (iii) the evaluation criterion
> (accuracy on a hidden test set); (iv) contributions and report outline.

## 2. Methodology
The work follows the CRISP-DM phases: business understanding, data understanding, data preparation,
modeling and evaluation.

### 2.1 Business understanding
* **Context.** Data published about products from different establishments; occasionally a product is sold on sale.
* **Goal.** From a set of product characteristics, classify whether a product is on discount (`y = 1`) or not (`y = 0`).
* **Success criterion.** Accuracy on the hidden test set (competition metric). Recall, precision and F-measure of the
  positive class (discount) are also reported to understand *how* each model errs.
* **Variables.** `i` is a record identifier (ignored for modeling, kept to build the submission); `d*` are discrete
  variables; `x*` are continuous variables; `y` is the target.
* **Limitation.** Variables are anonymised, so domain interpretation is limited to statistical patterns.
  > **TODO:** add any domain knowledge given in class.

### 2.2 Data understanding (EDA)

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                              GradientBoostingClassifier, RandomForestClassifier)
from sklearn.impute import KNNImputer, MissingIndicator, SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, RobustScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ------------------------------ configuration ------------------------------
SEED = 42
DATA_DIR = Path(".")
TRAIN_FILE = DATA_DIR / "training.csv"
TEST_FILE = DATA_DIR / "test.csv"
FIG_DIR, OUT_DIR = Path("figures"), Path("outputs")
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

N_SPLITS = 5          # outer cross-validation folds (model assessment)
INNER_SPLITS = 3      # inner folds used to tune hyper-parameters
TUNE = True           # False -> default hyper-parameters (much faster)
MAX_CAT_LEVELS = 20   # discrete variables with more levels are treated as numeric
RARE_CAT_FREQ = 0.01  # categories rarer than this are grouped into "infrequent"
CORR_THRESHOLD = 0.90 # |Pearson r| above which a numeric variable is redundant (see 2.3)
N_JOBS = -1


def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


def make_grid(n, ncols=3, w=4.2, h=3.2):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]:
        ax.set_visible(False)
    return fig, axes

#### Load data and basic inspection

In [ ]:
train = pd.read_csv(TRAIN_FILE)
print("Shape:", train.shape)
display(train.head())
train.info()

# The identifier must be unique; it is removed from the modeling data
assert train["i"].is_unique, "identifier 'i' is not unique"
train = train.drop(columns="i")

d_vars = [c for c in train.columns if c.startswith("d")]
x_vars = [c for c in train.columns if c.startswith("x")]
print(f"\n{len(d_vars)} discrete variables: {d_vars}")
print(f"{len(x_vars)} continuous variables: {x_vars}")

y = train["y"].astype(int)
X = train.drop(columns="y")

# discrete variables with few levels -> categorical (one-hot); many levels -> numeric
cat_cols = [c for c in d_vars if X[c].nunique() <= MAX_CAT_LEVELS]
num_cols = x_vars + [c for c in d_vars if c not in cat_cols]
print("\nTreated as categorical:", cat_cols)
print("Treated as numeric    :", num_cols)

#### Descriptive analysis

In [ ]:
# --- continuous variables
desc_x = train[x_vars].describe().T
desc_x["skew"] = train[x_vars].skew()
desc_x["kurtosis"] = train[x_vars].kurt()
desc_x["n_unique"] = train[x_vars].nunique()
desc_x["missing_%"] = 100 * train[x_vars].isna().mean()
display(desc_x.round(3))

# --- discrete variables
desc_d = pd.DataFrame({
    "n_unique": train[d_vars].nunique(),
    "missing_%": 100 * train[d_vars].isna().mean(),
    "mode": train[d_vars].mode().iloc[0],
    "mode_freq_%": [100 * train[c].value_counts(normalize=True).iloc[0] for c in d_vars],
    "min_cat_freq_%": [100 * train[c].value_counts(normalize=True).iloc[-1] for c in d_vars],
})
display(desc_d.round(2))

rare = {c: vc[vc < RARE_CAT_FREQ].index.tolist()
        for c in d_vars if (vc := train[c].value_counts(normalize=True)).lt(RARE_CAT_FREQ).any()}
print(f"Categories with frequency < {RARE_CAT_FREQ:.0%}:", rare or "none")

#### Target distribution

In [ ]:
counts = y.value_counts().sort_index()
props = y.value_counts(normalize=True).sort_index()
display(pd.DataFrame({"count": counts, "proportion": props.round(4)}))
print(f"Majority-class baseline accuracy: {props.max():.4f}")
print(f"Imbalance ratio (majority/minority): {counts.max() / counts.min():.2f}")

fig, ax = plt.subplots(figsize=(4, 3))
sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax, color="steelblue")
ax.set(xlabel="y (1 = discount)", ylabel="count", title="Target distribution")
savefig("target_distribution")
plt.show()

#### Histograms

In [ ]:
# Continuous variables: overall distribution
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.histplot(train[v].dropna(), kde=True, ax=ax, color="steelblue")
    ax.set_title(v)
fig.tight_layout()
savefig("hist_x")
plt.show()

# Continuous variables: distribution by class
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.histplot(data=train, x=v, hue="y", element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title(f"{v} by y")
fig.tight_layout()
savefig("hist_x_by_y")
plt.show()

In [ ]:
# Discrete variables: frequency and share of y=1 per level
fig, axes = make_grid(len(d_vars))
for ax, v in zip(axes, d_vars):
    if train[v].nunique() <= MAX_CAT_LEVELS:
        ct = pd.crosstab(train[v], train["y"])
        ct.plot(kind="bar", stacked=True, ax=ax, width=0.85, legend=(v == d_vars[0]))
    else:
        sns.histplot(train[v].dropna(), ax=ax, color="steelblue")
    ax.set_title(v)
fig.tight_layout()
savefig("hist_d")
plt.show()

for v in d_vars:
    print(f"\n{v} — share of y=1 per level")
    print(train.groupby(v)["y"].agg(n="size", pct_discount="mean").assign(
        pct_discount=lambda t: (100 * t["pct_discount"]).round(1)).T.to_string())

#### Boxplots

In [ ]:
# Raw boxplots (original scale)
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.boxplot(y=train[v], ax=ax, color="lightblue")
    ax.set_title(v)
fig.tight_layout()
savefig("box_x")
plt.show()

# All variables on a comparable (z-score) scale
Z = (train[x_vars] - train[x_vars].mean()) / train[x_vars].std()
fig, ax = plt.subplots(figsize=(1.0 * len(x_vars) + 3, 4))
sns.boxplot(data=Z, ax=ax, color="lightblue")
ax.set_title("Continuous variables (z-scores)")
savefig("box_x_standardised")
plt.show()

# By class
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.boxplot(data=train, x="y", y=v, ax=ax)
    ax.set_title(f"{v} by y")
fig.tight_layout()
savefig("box_x_by_y")
plt.show()

#### Correlations

In [ ]:
def high_corr_pairs(df, thr):
    corr = df.corr()
    cols = corr.columns
    pairs = [(cols[i], cols[j], corr.iloc[i, j])
             for i in range(len(cols)) for j in range(i + 1, len(cols))
             if abs(corr.iloc[i, j]) >= thr]
    out = pd.DataFrame(pairs, columns=["var_1", "var_2", "r"])
    return out.sort_values("r", key=lambda s: s.abs(), ascending=False)


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, method in zip(axes, ["pearson", "spearman"]):
    corr = train[x_vars + ["y"]].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7})
    ax.set_title(f"{method.capitalize()} correlation (x variables and y)")
fig.tight_layout()
savefig("correlations_x")
plt.show()

print("Pairs of continuous variables with |r| >= 0.7:")
display(high_corr_pairs(train[x_vars], 0.7).round(3))

corr_y = train[x_vars + ["y"]].corr()["y"].drop("y").sort_values(key=abs, ascending=False)
print("\nCorrelation of each continuous variable with y:")
display(corr_y.round(3).to_frame("r_with_y"))

In [ ]:
# Discrete variables: association with y and between each other (Cramér's V)
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    if min(ct.shape) < 2:
        return np.nan
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return np.sqrt((chi2 / ct.values.sum()) / (min(ct.shape) - 1))


assoc = pd.DataFrame({
    "cramers_v_with_y": [cramers_v(train[c], train["y"]) for c in d_vars],
    "chi2_p_value": [stats.chi2_contingency(pd.crosstab(train[c], train["y"]))[1] for c in d_vars],
}, index=d_vars)
display(assoc.sort_values("cramers_v_with_y", ascending=False).round(4))

if len(cat_cols) > 1:
    V = pd.DataFrame([[cramers_v(train[a], train[b]) for b in cat_cols] for a in cat_cols],
                     index=cat_cols, columns=cat_cols)
    fig, ax = plt.subplots(figsize=(0.7 * len(cat_cols) + 3, 0.6 * len(cat_cols) + 2.5))
    sns.heatmap(V, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax,
                annot_kws={"size": 7})
    ax.set_title("Association between discrete variables (Cramér's V)")
    savefig("cramers_v_d")
    plt.show()

#### Data quality overview: missing values and outliers

In [ ]:
missing = pd.DataFrame({"n_missing": train.isna().sum(), "pct_missing": (100 * train.isna().mean()).round(2)})
missing = missing[missing["n_missing"] > 0]
print("Variables with missing values:")
display(missing if len(missing) else "none")

# Is missingness related to the target?
for v in missing.index:
    print(f"\nShare of y=1 when {v} is missing vs. observed")
    print(train.groupby(train[v].isna())["y"].agg(["size", "mean"]).rename(
        index={True: "missing", False: "observed"}).round(3).to_string())


def tukey_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


outliers = pd.DataFrame({"n_outliers": [tukey_outliers(train[v]) for v in x_vars]}, index=x_vars)
outliers["pct_outliers"] = (100 * outliers["n_outliers"] / len(train)).round(2)
outliers["skew"] = train[x_vars].skew().round(2)
outliers["pct_zeros"] = (100 * (train[x_vars] == 0).mean()).round(2)
print("\nOutliers (Tukey 1.5 x IQR rule):")
display(outliers.sort_values("n_outliers", ascending=False))

> **TODO — EDA narrative.** Summarise: class balance; which variables are skewed / have outliers / many zeros
> (are they errors or genuine extremes?); which variables are informative (class-conditional
> histograms/boxplots, correlation with `y`, Cramér's V); redundant pairs; rare categories; missing-value pattern.

### 2.3 Data preparation
All preparation steps are implemented **inside a pipeline that is fitted on the training folds only**
(imputation values, clipping fences, scaling parameters, one-hot categories, redundancy filter and resampling are
learned from each training fold and merely applied to the validation fold / test set). This avoids data leakage.

Each decision below is supported by a small experiment: two reference classifiers (kNN, sensitive to scale/outliers,
and Random Forest, robust to them) are evaluated with the stratified cross-validation of section *Cross-validation data*.
The options are chosen **greedily, one step at a time**, by mean accuracy (the competition metric); recall/precision/F-measure
are shown so the choice can be discussed. Any choice can be overridden in the configuration cell.

In [ ]:
# ---------------------------- custom transformers ----------------------------
class TukeyClipper(BaseEstimator, TransformerMixin):
    """Clips each column to [Q1 - k*IQR, Q3 + k*IQR] computed on the training data (winsorising)."""

    def __init__(self, k=1.5):
        self.k = k

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.nanpercentile(X, 25, axis=0), np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_, self.upper_ = q1 - self.k * iqr, q3 + self.k * iqr
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


class CorrelationFilter(BaseEstimator, TransformerMixin):
    """Drops a column when |Pearson r| with an already-kept column exceeds `threshold`."""

    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        corr = np.abs(np.atleast_2d(np.corrcoef(X, rowvar=False)))
        keep = []
        for j in range(corr.shape[0]):
            if not any(corr[i, j] > self.threshold for i in keep):
                keep.append(j)
        self.keep_ = np.array(keep)
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.keep_]


def make_preprocessor(missing="median", outliers="keep", corr_threshold=None):
    imputer = {"median": SimpleImputer(strategy="median"),
               "mean": SimpleImputer(strategy="mean"),
               "knn": KNNImputer(n_neighbors=5)}[missing]
    steps = [("imputer", imputer)]
    if outliers == "clip":
        steps += [("clip", TukeyClipper()), ("scale", StandardScaler())]
    elif outliers == "power":
        steps += [("power", PowerTransformer(method="yeo-johnson"))]
    elif outliers == "robust":
        steps += [("scale", RobustScaler())]
    elif outliers == "keep":
        steps += [("scale", StandardScaler())]
    else:
        raise ValueError(outliers)
    if corr_threshold is not None:
        steps.append(("corr", CorrelationFilter(corr_threshold)))

    blocks = [
        ("num", SkPipeline(steps), num_cols),
        # flags the rows whose values were imputed (only for columns that had NaN in training)
        ("miss_flag", MissingIndicator(features="missing-only", error_on_new=False), x_vars),
    ]
    if cat_cols:
        blocks.append(("cat", SkPipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                     min_frequency=RARE_CAT_FREQ, sparse_output=False)),
        ]), cat_cols))
    return ColumnTransformer(blocks, remainder="drop")


def build_pipeline(model, missing="median", outliers="keep", balancing="none", corr_threshold=None):
    steps = [("prep", make_preprocessor(missing, outliers, corr_threshold))]
    if balancing == "smote":
        steps.append(("sampler", SMOTE(random_state=SEED)))
    elif balancing == "undersample":
        steps.append(("sampler", RandomUnderSampler(random_state=SEED)))
    elif balancing != "none":
        raise ValueError(balancing)
    steps.append(("clf", clone(model)))
    return ImbPipeline(steps)

#### Cross-validation data
A **stratified** k-fold split keeps the class proportions in every fold. The same folds are used in every experiment
and for every model, so results are directly comparable.

In [ ]:
outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_of = np.zeros(len(X), dtype=int)
rows = []
for k, (tr, va) in enumerate(outer_cv.split(X, y), start=1):
    fold_of[va] = k
    rows.append({"fold": k, "n_train": len(tr), "n_validation": len(va),
                 "%y=1 train": round(100 * y.iloc[tr].mean(), 2),
                 "%y=1 validation": round(100 * y.iloc[va].mean(), 2)})
display(pd.DataFrame(rows).set_index("fold"))
pd.DataFrame({"row": np.arange(len(X)), "fold": fold_of}).to_csv(OUT_DIR / "cv_folds.csv", index=False)

REF_MODELS = {"kNN": KNeighborsClassifier(n_neighbors=11),
              "RandomForest": RandomForestClassifier(n_estimators=200, random_state=SEED)}
METRICS = ["accuracy", "recall", "precision", "f1"]


def compare(options, base):
    """Cross-validates the reference models under each option; returns (long table, accuracy summary)."""
    rows = []
    for label, kw in options.items():
        cfg = {**base, **kw}
        for name, model in REF_MODELS.items():
            res = cross_validate(build_pipeline(model, **cfg), X, y, cv=outer_cv,
                                 scoring=METRICS, n_jobs=N_JOBS)
            rows.append({"option": label, "model": name, **{m: res[f"test_{m}"].mean() for m in METRICS}})
    long = pd.DataFrame(rows)
    acc = long.pivot(index="option", columns="model", values="accuracy")
    acc["mean_accuracy"] = acc.mean(axis=1)
    return long, acc.loc[list(options)]


def show(long, acc):
    display(acc.round(4))
    display(long.pivot(index="option", columns="model", values=["recall", "precision", "f1"]).round(4))


def pick(acc, override=None):
    return override if override is not None else acc["mean_accuracy"].idxmax()


BASE = dict(missing="median", outliers="keep", balancing="none", corr_threshold=None)

#### Missing values
Two questions: (1) remove the incomplete rows or impute them? (2) which imputation?

Removing rows loses information, and — crucially — **the test set must be predicted for every `i`**, so rows with missing
values cannot be dropped there. The experiment compares both strategies on the *same* validation rows (complete rows),
and shows the accuracy of the imputing pipeline on the incomplete rows, which the "drop" strategy cannot score at all.

In [ ]:
incomplete = X.isna().any(axis=1)
print(f"Rows with at least one missing value: {incomplete.sum()} ({100 * incomplete.mean():.2f}% of training data)")

if incomplete.any():
    ref = REF_MODELS["RandomForest"]
    res = {"drop incomplete rows (complete val. rows)": [],
           "median imputation (same complete val. rows)": [],
           "median imputation (incomplete val. rows only)": []}
    for tr, va in outer_cv.split(X, y):
        Xtr, ytr, Xva, yva = X.iloc[tr], y.iloc[tr], X.iloc[va], y.iloc[va]
        ok_tr, ok_va = ~Xtr.isna().any(axis=1), ~Xva.isna().any(axis=1)
        p = build_pipeline(ref).fit(Xtr[ok_tr], ytr[ok_tr])
        res["drop incomplete rows (complete val. rows)"].append(accuracy_score(yva[ok_va], p.predict(Xva[ok_va])))
        p = build_pipeline(ref).fit(Xtr, ytr)
        pred = pd.Series(p.predict(Xva), index=Xva.index)
        res["median imputation (same complete val. rows)"].append(accuracy_score(yva[ok_va], pred[ok_va]))
        if (~ok_va).any():
            res["median imputation (incomplete val. rows only)"].append(accuracy_score(yva[~ok_va], pred[~ok_va]))
    display(pd.Series({k: np.mean(v) for k, v in res.items()}, name="mean accuracy (Random Forest)").round(4).to_frame())

    cmp_missing, acc_missing = compare({"median": {"missing": "median"},
                                        "mean": {"missing": "mean"},
                                        "knn": {"missing": "knn"}}, BASE)
    show(cmp_missing, acc_missing)
    best_missing = pick(acc_missing)
else:
    print("No missing values in the training data: simple median imputation stays in the pipeline as a safeguard "
          "for the test set.")
    best_missing = "median"
print("-> missing-value strategy:", best_missing)

#### Outliers
Observations flagged by the Tukey rule are **kept** (they are not shown to be errors and removing rows would
lose information, and the test set cannot be filtered either). Instead, four treatments of their *influence* are compared:
keep as is (z-score scaling), clip at the Tukey fences (winsorising), robust scaling (median/IQR) and a Yeo-Johnson power
transform (also reduces skew).

In [ ]:
cmp_out, acc_out = compare({"keep": {"outliers": "keep"},
                            "clip": {"outliers": "clip"},
                            "robust": {"outliers": "robust"},
                            "power": {"outliers": "power"}}, {**BASE, "missing": best_missing})
show(cmp_out, acc_out)
best_outliers = pick(acc_out)
print("-> outlier treatment:", best_outliers)

#### Unbalanced data
If the classes are imbalanced, a model can reach a high accuracy by favouring the majority class.
Compared: no treatment, SMOTE oversampling of the minority class, and random undersampling of the majority class —
applied **only to the training folds** (the validation folds keep the real class distribution).

Note: SMOTE after one-hot encoding creates fractional dummy values, a known limitation.

In [ ]:
print(y.value_counts(normalize=True).sort_index().round(4).to_string())
cmp_bal, acc_bal = compare({"none": {"balancing": "none"},
                            "smote": {"balancing": "smote"},
                            "undersample": {"balancing": "undersample"}},
                           {**BASE, "missing": best_missing, "outliers": best_outliers})
show(cmp_bal, acc_bal)
best_balancing = pick(acc_bal)
print("-> balancing:", best_balancing)

#### Feature redundancy
Highly correlated continuous variables carry nearly the same information. A pair is considered redundant above the
threshold in the configuration (|r| > 0.90 by default); the later variable of the pair is dropped inside the pipeline.
Discrete variables are checked with Cramér's V (heatmap above); rare categories are grouped by the one-hot encoder.

In [ ]:
x_imputed = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X[num_cols]), columns=num_cols)
pairs = high_corr_pairs(x_imputed, CORR_THRESHOLD)
print(f"Pairs with |r| >= {CORR_THRESHOLD}:")
display(pairs.round(3) if len(pairs) else "none")
dropped = [c for k, c in enumerate(num_cols) if k not in CorrelationFilter(CORR_THRESHOLD).fit(x_imputed.values).keep_]
print("Variables dropped by the filter on the full training data:", dropped or "none")

# near-constant variables (low variability)
near_const = X.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0]).sort_values(ascending=False)
print("\nShare of the most frequent value per variable (top 5):")
print(near_const.head(5).round(4).to_string())

cmp_corr, acc_corr = compare({"no filter": {"corr_threshold": None},
                              "0.95": {"corr_threshold": 0.95},
                              "0.90": {"corr_threshold": 0.90},
                              "0.80": {"corr_threshold": 0.80}},
                             {"missing": best_missing, "outliers": best_outliers, "balancing": best_balancing})
show(cmp_corr, acc_corr)
best_corr = {"no filter": None, "0.95": 0.95, "0.90": 0.90, "0.80": 0.80}[pick(acc_corr)]
print("-> correlation threshold:", best_corr)

#### Final preparation configuration

In [ ]:
# Set a value to override the automatic choice (e.g. OVERRIDE["outliers"] = "clip")
OVERRIDE = {"missing": None, "outliers": None, "balancing": None, "corr_threshold": "auto"}

PREP = dict(
    missing=OVERRIDE["missing"] or best_missing,
    outliers=OVERRIDE["outliers"] or best_outliers,
    balancing=OVERRIDE["balancing"] or best_balancing,
    corr_threshold=best_corr if OVERRIDE["corr_threshold"] == "auto" else OVERRIDE["corr_threshold"],
)
print("Preparation used for every model:", PREP)

Xt = make_preprocessor(PREP["missing"], PREP["outliers"], PREP["corr_threshold"]).fit_transform(X, y)
print(f"Design matrix delivered to the models: {Xt.shape[0]} rows x {Xt.shape[1]} columns "
      f"(from {X.shape[1]} original variables)")

> **TODO — data-preparation narrative.** For each step (missing values, outliers, unbalanced data, redundancy) state the
> finding, the options compared (tables above), the choice and the reason.

### 2.4 Modeling
Eight classifiers plus a majority-class baseline, all inside the same preparation pipeline:

| Family | Models |
|---|---|
| Instance-based | k-Nearest Neighbors |
| Tree | Decision Tree |
| Probabilistic | Gaussian Naive Bayes |
| Neural network | Multilayer Perceptron (MLP) |
| Ensemble — bagging | Random Forest, Bagging of decision trees |
| Ensemble — boosting | AdaBoost, Gradient Boosting |

Hyper-parameters are tuned with **nested cross-validation**: a grid search with `INNER_SPLITS` folds inside every
training set of the `N_SPLITS` outer folds, optimising accuracy. The outer folds are never used for tuning, so the
reported scores are not optimistically biased.

In [ ]:
MODELS = {
    "Baseline (majority class)": (DummyClassifier(strategy="most_frequent"), {}),
    "kNN": (KNeighborsClassifier(),
            {"clf__n_neighbors": [3, 5, 7, 11, 15, 21, 31], "clf__weights": ["uniform", "distance"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=SEED),
                      {"clf__max_depth": [3, 5, 8, 12, None], "clf__min_samples_leaf": [1, 5, 20]}),
    "Naive Bayes": (GaussianNB(),
                    {"clf__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}),
    "MLP": (MLPClassifier(max_iter=500, early_stopping=True, random_state=SEED),
            {"clf__hidden_layer_sizes": [(32,), (64,), (64, 32)], "clf__alpha": [1e-4, 1e-2]}),
    "Random Forest": (RandomForestClassifier(random_state=SEED),
                      {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 10, 20],
                       "clf__min_samples_leaf": [1, 3]}),
    "Bagging (trees)": (BaggingClassifier(estimator=DecisionTreeClassifier(random_state=SEED), random_state=SEED),
                        {"clf__n_estimators": [50, 150], "clf__max_samples": [0.7, 1.0]}),
    "AdaBoost": (AdaBoostClassifier(random_state=SEED),
                 {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.05, 0.3, 1.0]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=SEED),
                          {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.05, 0.1],
                           "clf__max_depth": [2, 3]}),
}


def metrics_of(y_true, y_pred):
    """Metrics for the positive class (y = 1, discount)."""
    return {"accuracy": accuracy_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0)}


def evaluate_model(estimator, grid):
    """Nested CV: returns per-fold metrics, out-of-fold predictions and the selected hyper-parameters."""
    fold_rows, best_params = [], []
    oof = pd.Series(index=X.index, dtype=int)
    inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    for k, (tr, va) in enumerate(outer_cv.split(X, y), start=1):
        pipe = build_pipeline(estimator, **PREP)
        if TUNE and grid:
            model = GridSearchCV(pipe, grid, cv=inner_cv, scoring="accuracy", n_jobs=N_JOBS)
            model.fit(X.iloc[tr], y.iloc[tr])
            best_params.append({p.replace("clf__", ""): v for p, v in model.best_params_.items()})
        else:
            model = pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = model.predict(X.iloc[va])
        oof.iloc[va] = pred
        fold_rows.append({"fold": k, **metrics_of(y.iloc[va], pred)})
    return pd.DataFrame(fold_rows).set_index("fold"), oof, best_params

In [ ]:
import time

fold_scores, oof_preds, tuned_params = {}, {}, {}
for name, (estimator, grid) in MODELS.items():
    t0 = time.time()
    fold_scores[name], oof_preds[name], tuned_params[name] = evaluate_model(estimator, grid)
    print(f"{name:<28} accuracy = {fold_scores[name]['accuracy'].mean():.4f} "
          f"(+/- {fold_scores[name]['accuracy'].std():.4f})   [{time.time() - t0:5.0f}s]")

### 2.5 Evaluation
* **Protocol.** Nested, stratified cross-validation (outer `N_SPLITS` folds for assessment, inner `INNER_SPLITS` folds for tuning);
  the same outer folds for every model. Reported values are the **mean (± standard deviation) over the outer folds**.
* **Metrics** (positive class: `y = 1`, product on discount), with TP, TN, FP, FN from the confusion matrix:
  * Accuracy = (TP + TN) / (TP + TN + FP + FN) — the competition metric.
  * Precision = TP / (TP + FP) — of the products predicted as discounted, how many truly are.
  * Recall = TP / (TP + FN) — of the truly discounted products, how many are found.
  * F-measure = 2 · Precision · Recall / (Precision + Recall).
* **Reference.** A majority-class baseline shows the accuracy that is achievable without any model.
* **Final model.** The model with the highest mean accuracy is refitted on the whole training set (tuned with
  `N_SPLITS`-fold CV) and used to predict the hidden test set.

## 3. Results

In [ ]:
def summarise(fold_scores):
    mean = pd.DataFrame({m: s[METRICS].mean() for m, s in fold_scores.items()}).T
    std = pd.DataFrame({m: s[METRICS].std() for m, s in fold_scores.items()}).T
    mean = mean.sort_values("accuracy", ascending=False)
    std = std.loc[mean.index]
    nice = {"accuracy": "Accuracy", "recall": "Recall", "precision": "Precision", "f1": "F-measure"}
    numeric = mean.rename(columns=nice)[["Accuracy", "Recall", "Precision", "F-measure"]]
    formatted = numeric.copy().astype(object)
    for c, c_raw in zip(numeric.columns, ["accuracy", "recall", "precision", "f1"]):
        formatted[c] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(mean[c_raw], std[c_raw])]
    numeric.index.name = formatted.index.name = "Model"
    return numeric, formatted


results_numeric, results_fmt = summarise(fold_scores)
display(results_fmt)

results_numeric.round(4).to_csv(OUT_DIR / "model_comparison.csv")
try:
    (OUT_DIR / "model_comparison.md").write_text(results_fmt.reset_index().to_markdown(index=False))
except ImportError:
    print("tabulate not installed: markdown export skipped (pip install tabulate)")

In [ ]:
# Accuracy per outer fold, and all metrics side by side
order = results_numeric.index.tolist()
acc_folds = pd.DataFrame({m: fold_scores[m]["accuracy"] for m in order})

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.1, 1]})
sns.boxplot(data=acc_folds, orient="h", ax=axes[0], color="lightblue")
sns.stripplot(data=acc_folds, orient="h", ax=axes[0], color="black", size=4)
axes[0].axvline(y.value_counts(normalize=True).max(), color="red", ls="--", lw=1, label="majority-class baseline")
axes[0].set(xlabel="accuracy per outer fold", title="Accuracy across folds")
axes[0].legend(loc="lower right")

results_numeric.plot(kind="barh", ax=axes[1], width=0.85)
axes[1].invert_yaxis()
axes[1].set(xlim=(0, 1), title="Mean metrics by model", ylabel="")
axes[1].legend(loc="lower right", fontsize=8)
fig.tight_layout()
savefig("model_comparison")
plt.show()

In [ ]:
# Best model (excluding the baseline): confusion matrix from the out-of-fold predictions and tuned hyper-parameters
best_name = [m for m in results_numeric.index if not m.startswith("Baseline")][0]
print("Best model:", best_name)

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix(y, oof_preds[best_name]), display_labels=["no discount", "discount"]).plot(
    ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"{best_name} — out-of-fold confusion matrix")
savefig("confusion_matrix_best")
plt.show()

if TUNE:
    hp = pd.DataFrame({m: [", ".join(f"{k}={v}" for k, v in p.items()) for p in tuned_params[m]]
                       for m in order if tuned_params[m]}).T
    hp.columns = [f"fold {k}" for k in range(1, N_SPLITS + 1)]
    pd.set_option("display.max_colwidth", 120)
    display(hp)

#### Final model and submission

In [ ]:
est, grid = MODELS[best_name]
final_pipe = build_pipeline(est, **PREP)
final_model = GridSearchCV(final_pipe, grid, cv=StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED),
                           scoring="accuracy", n_jobs=N_JOBS).fit(X, y) if (TUNE and grid) else final_pipe.fit(X, y)
if TUNE and grid:
    print("Selected hyper-parameters:", final_model.best_params_, f"(CV accuracy {final_model.best_score_:.4f})")

if TEST_FILE.exists():
    test = pd.read_csv(TEST_FILE)
    submission = pd.DataFrame({"i": test["i"], "y": final_model.predict(test[X.columns]).astype(int)})
    submission.to_csv(OUT_DIR / "submission.csv", index=False)
    assert list(submission.columns) == ["i", "y"] and submission["i"].is_unique
    assert set(submission["y"]) <= {0, 1} and len(submission) == len(test)
    print(f"submission.csv written: {len(submission)} rows; predicted share of y=1 = {submission['y'].mean():.3f}"
          f" (training: {y.mean():.3f})")
    display(submission.head())
else:
    print(f"{TEST_FILE} not found: submission not created.")

In [ ]:
# Which variables drive the best model? (permutation importance on an untouched 20% hold-out)
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
best_params = final_model.best_params_ if (TUNE and grid) else {}
probe = build_pipeline(est, **PREP).set_params(**best_params).fit(Xa, ya)
imp = permutation_importance(probe, Xb, yb, scoring="accuracy", n_repeats=10, random_state=SEED, n_jobs=N_JOBS)
imp = pd.Series(imp.importances_mean, index=X.columns).sort_values()
print(f"Hold-out accuracy of the probe model: {accuracy_score(yb, probe.predict(Xb)):.4f}")

fig, ax = plt.subplots(figsize=(5, 0.28 * len(imp) + 1.5))
imp.plot(kind="barh", ax=ax, color="steelblue")
ax.set(xlabel="drop in accuracy when the variable is shuffled", title=f"Permutation importance — {best_name}")
savefig("permutation_importance")
plt.show()

> **TODO — results narrative.** Compare models (table and figures above): which family works best and by how much over
> the majority-class baseline; stability across folds (standard deviation); trade-off between recall and precision;
> effect of the preparation choices; the most important variables.

## 4. Conclusions
Summary table (mean ± standard deviation over the outer folds; positive class = discount):

In [ ]:
display(results_fmt)
print(f"Best model: {best_name} — accuracy {results_numeric.loc[best_name, 'Accuracy']:.4f}, "
      f"recall {results_numeric.loc[best_name, 'Recall']:.4f}, "
      f"precision {results_numeric.loc[best_name, 'Precision']:.4f}, "
      f"F-measure {results_numeric.loc[best_name, 'F-measure']:.4f}")

> **TODO — conclusions.** (i) answer the business question: can discount status be predicted from these variables, and how well?
> (ii) best model and why; (iii) limitations (anonymised variables, single dataset, accuracy as the only competition metric);
> (iv) future work (feature engineering, stacking/voting ensembles, probability calibration and threshold tuning,
> additional hyper-parameter search).